# Banking77: TF-IDF baseline и разбор ошибок

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Задача

Обучите прозрачное базовое решение для классификации банковских обращений по 77 намерениям. Здесь важна не только итоговая метрика: найдите классы, которые модель путает, прочитайте конкретные ошибки и предложите следующий эксперимент.

Официальную обучающую часть разделите на обучающую и проверочную выборки. Официальную тестовую часть откройте один раз после фиксации конвейера и правил анализа.

**Что должно получиться:** честное сравнение словных n-грамм длиной 1–2 и символьных `char_wb` n-грамм при одинаковом бюджете подбора, логистическая регрессия, доля правильных ответов (accuracy) и macro-F1, полнота по каждому классу, пять наиболее частых пар ошибок, ручная типология минимум 20 ошибок и один обоснованный следующий эксперимент.

Источник: Banking77 из официального репозитория PolyAI, ревизия закреплена в `data/datasets.json`, лицензия CC BY 4.0. CPU достаточно; API-ключи не нужны. В локальной копии репозитория используются сохранённые данные. Если открыть только notebook в Colab, подготовительная ячейка один раз скачает три официальных файла из закреплённого commit, проверит SHA-256 и дальше будет работать локально.

In [ ]:
import hashlib
import json
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


BANKING_FILES = {
    "train.csv": (
        "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/train.csv",
        "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b",
    ),
    "test.csv": (
        "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/test.csv",
        "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d",
    ),
    "categories.json": (
        "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/categories.json",
        "53261da888122daf2d120d925458631d9619e15d82e56052e7a42e535ce32b63",
    ),
}


def sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def require_sha256(payload: bytes, expected: str, label: str) -> None:
    actual = sha256(payload)
    if actual != expected:
        raise RuntimeError(f"{label}: checksum changed: {actual}")


def require_final_hostname(response, expected: str) -> None:
    actual = urlparse(response.geturl()).hostname
    if actual != expected:
        raise RuntimeError(f"unexpected redirect host: {actual}")


def resolve_banking_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        directory = candidate / "data" / "banking77"
        if all((directory / name).is_file() for name in BANKING_FILES):
            for name, spec in BANKING_FILES.items():
                require_sha256((directory / name).read_bytes(), spec[1], f"Banking77 {name}")
            return directory

    directory = Path("/tmp/ml-mentor-labs-datasets-v1/banking77")
    directory.mkdir(parents=True, exist_ok=True)
    for name, (url, expected_sha256) in BANKING_FILES.items():
        path = directory / name
        if path.is_file():
            try:
                require_sha256(path.read_bytes(), expected_sha256, f"cached Banking77 {name}")
                continue
            except RuntimeError:
                path.unlink()
        request = urllib.request.Request(url, headers={"User-Agent": "ML-Mentor-Labs/1.0"})
        with urllib.request.urlopen(request, timeout=120) as response:
            require_final_hostname(response, "raw.githubusercontent.com")
            payload = response.read()
        require_sha256(payload, expected_sha256, f"Banking77 {name}")
        path.write_bytes(payload)
    return directory


DATA_DIR = resolve_banking_dir()
OFFICIAL_TEST_PATH = DATA_DIR / "test.csv"
official_train = pd.read_csv(DATA_DIR / "train.csv")
categories = json.loads((DATA_DIR / "categories.json").read_text(encoding="utf-8"))

train_df, validation_df = train_test_split(
    official_train,
    test_size=0.2,
    stratify=official_train["category"],
    random_state=42,
)
assert official_train["category"].nunique() == len(categories) == 77
assert set(train_df.index).isdisjoint(validation_df.index)


## 1. Сравните два представления текста

Соберите два семейства конвейеров (`Pipeline`) из `TfidfVectorizer` и `LogisticRegression`: словные n-граммы длиной 1–2 и символьные `char_wb` n-граммы длиной 3–5. Назовите шаги `tfidf` и `classifier`. В обоих вариантах используйте `min_df=2`, `max_features=50_000`, `sublinear_tf=True` и логистическую регрессию с `max_iter=1000`, `random_state=42`. Меняется только представление текста и одно из двух значений `C`. Это одинаковый бюджет подбора: по два обучения на вариант, одно разбиение и одна метрика выбора. Все решения принимайте по macro-F1 на проверочной выборке; при равенстве используйте accuracy, затем имя варианта в алфавитном порядке и меньшее `C`. Не подглядывайте в тестовую выборку.

In [ ]:
VARIANTS = ("word_1_2", "char_wb")
C_VALUES = (0.5, 2.0)


def build_intent_model(variant: str, *, c: float) -> Pipeline:
    # TODO: соберите TF-IDF + Logistic Regression pipeline для указанного variant.
    raise NotImplementedError


# TODO: выполните ровно len(C_VALUES) fit на каждый variant и сохраните все результаты.
candidate_results = pd.DataFrame(columns=["variant", "c", "accuracy", "macro_f1"])
candidate_models = {}
# TODO: выберите одну конфигурацию по validation macro-F1 и получите validation_pred.
SELECTED_VARIANT = None
SELECTED_C = None
model = None
validation_pred = None
validation_metrics = None


## 2. Разберите ошибки

Macro-F1 одинаково учитывает редкие и частые классы, поэтому не ограничивайтесь accuracy. Постройте таблицу полноты (recall) по классам и матрицу ошибок без диагонали. Затем прочитайте ошибочные тексты и присвойте им понятные категории, например: близкие намерения, слишком короткий запрос, неоднозначность или лексика, которой не было в обучающей выборке.

In [ ]:
def per_class_recall(y_true: pd.Series, y_pred: np.ndarray) -> pd.DataFrame:
    # TODO: верните columns=[category, support, recall], по одному ряду на класс.
    raise NotImplementedError


def top_confusions(y_true: pd.Series, y_pred: np.ndarray, *, limit: int = 5) -> pd.DataFrame:
    # TODO: верните самые частые ошибки columns=[actual, predicted, count], без диагонали.
    raise NotImplementedError


recall_table = None
confusion_pairs = None

# TODO: отберите минимум 20 ошибочных validation-примеров и заполните error_type и note.
error_audit = pd.DataFrame(columns=["text", "actual", "predicted", "error_type", "note"])
NEXT_EXPERIMENT = None


## 3. Одна финальная проверка на тестовой выборке

Когда конвейер, метрики и типология ошибок зафиксированы, прочитайте официальную тестовую часть, переобучите модель на полной официальной обучающей части и посчитайте test accuracy и macro-F1. После этого не меняйте решение по тестовому результату.

In [ ]:
# TODO: выполните этот блок только после фиксации решения.
official_test = None
final_model = None
test_pred = None
test_metrics = None


In [ ]:
# Self-check: запускайте после выполнения всех TODO.
assert model is not None and validation_pred is not None
assert len(validation_pred) == len(validation_df)
assert len(candidate_results) == len(VARIANTS) * len(C_VALUES)
assert set(candidate_results["variant"]) == set(VARIANTS)
assert candidate_results.groupby("variant").size().eq(len(C_VALUES)).all()
assert set(candidate_results["c"]) == set(C_VALUES)
expected_candidates = {(variant, c) for variant in VARIANTS for c in C_VALUES}
assert set(candidate_models) == expected_candidates
assert all({"tfidf", "classifier"} <= set(candidate.named_steps) for candidate in candidate_models.values())
assert all(candidate.named_steps["tfidf"].min_df == 2 for candidate in candidate_models.values())
assert all(candidate.named_steps["tfidf"].max_features == 50_000 for candidate in candidate_models.values())
assert all(candidate.named_steps["tfidf"].sublinear_tf is True for candidate in candidate_models.values())
assert all(candidate.named_steps["classifier"].max_iter == 1000 for candidate in candidate_models.values())
assert all(candidate.named_steps["classifier"].random_state == 42 for candidate in candidate_models.values())
assert all(np.isclose(candidate.named_steps["classifier"].C, c) for (_, c), candidate in candidate_models.items())
assert candidate_models[("word_1_2", C_VALUES[0])].named_steps["tfidf"].analyzer == "word"
assert candidate_models[("word_1_2", C_VALUES[0])].named_steps["tfidf"].ngram_range == (1, 2)
assert candidate_models[("char_wb", C_VALUES[0])].named_steps["tfidf"].analyzer == "char_wb"
assert candidate_models[("char_wb", C_VALUES[0])].named_steps["tfidf"].ngram_range == (3, 5)
assert candidate_results[["accuracy", "macro_f1"]].apply(lambda column: column.between(0, 1).all()).all()
assert SELECTED_VARIANT in VARIANTS and SELECTED_C in C_VALUES
expected_best = candidate_results.sort_values(
    ["macro_f1", "accuracy", "variant", "c"],
    ascending=[False, False, True, True],
).iloc[0]
assert (SELECTED_VARIANT, SELECTED_C) == (expected_best["variant"], expected_best["c"])
assert model is candidate_models[(SELECTED_VARIANT, SELECTED_C)]
assert set(validation_metrics) == {"accuracy", "macro_f1"}
assert 0 <= validation_metrics["accuracy"] <= 1
assert 0 <= validation_metrics["macro_f1"] <= 1
assert recall_table is not None and set(recall_table.columns) == {"category", "support", "recall"}
assert len(recall_table) == 77 and recall_table["recall"].between(0, 1).all()
assert confusion_pairs is not None and set(confusion_pairs.columns) == {"actual", "predicted", "count"}
assert len(confusion_pairs) == 5
assert (confusion_pairs["actual"] != confusion_pairs["predicted"]).all()
assert confusion_pairs["count"].gt(0).all()
assert confusion_pairs["count"].tolist() == sorted(confusion_pairs["count"], reverse=True)
assert len(error_audit) >= 20
assert error_audit[["error_type", "note"]].notna().all().all()
assert (error_audit["actual"] != error_audit["predicted"]).all()
assert isinstance(NEXT_EXPERIMENT, str) and NEXT_EXPERIMENT.strip()
assert official_test is not None and final_model is not None and test_pred is not None
assert len(test_pred) == len(official_test)
assert set(test_metrics) == {"accuracy", "macro_f1"}
print({"validation": validation_metrics, "test": test_metrics})
